# Phase 3b — Additional Text Encoders and Lyric Chunking

Revision steps 8 (more text encoders) and the lyric part of 8b from `NEXT_STEPS.md`.

**Chunking.** Phase 3 passed each lyric to `all-mpnet-base-v2` in one piece; sentence-transformers silently truncates
at 384 tokens, which cuts off 23% of MERGE lyrics (14% exceed 512). Here every encoder sees the whole lyric: lines are
packed greedily into chunks that fit the model's limit (a line longer than the limit is split by tokens), each chunk is
embedded, and the track vector is the token-weighted mean of the L2-normalized chunk embeddings.

| Name | Model | Max tokens | Output |
|---|---|---|---|
| `sbert_mpnet_trunc` | `all-mpnet-base-v2`, Phase 3 vector (truncated) | 384 | 768 |
| `sbert_mpnet_chunk` | `all-mpnet-base-v2`, chunked | 384 | 768 |
| `minilm_chunk` | `all-MiniLM-L6-v2`, chunked | 256 | 384 |
| `bge_chunk` | `BAAI/bge-base-en-v1.5`, chunked (no query instruction: symmetric similarity) | 512 | 768 |
| `emoroberta_emb` | `j-hartmann/emotion-english-distilroberta-base`, mean-pooled last hidden state | 128 | 768 |
| `emoroberta_probs` | same model, softmax over its 7 emotion classes | 128 | 7 |
| `lexicon` | Phase 3 NRC (10) + VADER compound + lexical diversity | — | 12 |

The emotion classifier was fine-tuned on short texts, so it gets 128-token chunks. Its 7 classes (anger, disgust,
fear, joy, neutral, sadness, surprise) are discrete emotions, not MERGE quadrants: no MERGE labels are used.

**Output:** `outputs/encoders/text_<name>.npy`, rows in the order of `outputs/encoders/text_song_ids.csv`
(= `lyrics_features.parquet` order).

In [1]:
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

warnings.filterwarnings('ignore')
torch.set_num_threads(8)

ROOT = os.environ.get('MUSIC_RESEARCH_ROOT', os.path.abspath('.'))
OUT_DIR = os.path.join(ROOT, 'outputs')
ENC_DIR = os.path.join(OUT_DIR, 'encoders')
os.makedirs(ENC_DIR, exist_ok=True)

lyrics_df = pd.read_parquet(os.path.join(OUT_DIR, 'lyrics_features.parquet'))
merged    = pd.read_parquet(os.path.join(OUT_DIR, 'merged_dataset.parquet')).set_index('song_id')
sids  = lyrics_df['song_id'].tolist()
texts = merged.loc[sids, 'lyrics_text'].tolist()
lyrics_df[['song_id']].to_csv(os.path.join(ENC_DIR, 'text_song_ids.csv'), index=False)
print(f'{len(texts)} lyrics')

def l2(X):
    return X / np.linalg.norm(X, axis=-1, keepdims=True)

def chunk(text, tok, budget):
    # Greedy line packing into chunks of <= budget tokens (no special tokens counted)
    chunks, cur, cur_n = [], [], 0
    for line in [l.strip() for l in text.split('\n') if l.strip()]:
        ids = tok(line, add_special_tokens=False).input_ids
        if len(ids) > budget:                                  # very long line: split by tokens
            if cur: chunks.append((' \n'.join(cur), cur_n)); cur, cur_n = [], 0
            for s in range(0, len(ids), budget):
                piece = ids[s:s + budget]; chunks.append((tok.decode(piece), len(piece)))
            continue
        if cur_n + len(ids) > budget and cur:
            chunks.append(('\n'.join(cur), cur_n)); cur, cur_n = [], 0
        cur.append(line); cur_n += len(ids)
    if cur:
        chunks.append(('\n'.join(cur), cur_n))
    return chunks or [(text[:1000], 1)]

2075 lyrics


## 1. Sentence encoders (chunked)

In [2]:
def st_chunked(model_name, out_name):
    path = os.path.join(ENC_DIR, f'text_{out_name}.npy')
    if os.path.exists(path):
        print(f'{out_name}: exists'); return np.load(path)
    m = SentenceTransformer(model_name, device='cpu')
    budget = m.max_seq_length - 2
    t0 = time.time()
    all_chunks = [chunk(t, m.tokenizer, budget) for t in texts]
    flat = [c for cs in all_chunks for c, _ in cs]
    E = m.encode(flat, batch_size=32, normalize_embeddings=True, show_progress_bar=False)
    out, k = [], 0
    for cs in all_chunks:
        w = np.array([n for _, n in cs], float)
        out.append((E[k:k + len(cs)] * w[:, None]).sum(0) / w.sum()); k += len(cs)
    out = np.vstack(out).astype(np.float32)
    np.save(path, out)
    n_ch = np.array([len(cs) for cs in all_chunks])
    print(f'{out_name}: {out.shape}, max_seq={m.max_seq_length}, chunks/lyric mean {n_ch.mean():.2f}, '
          f'{(n_ch > 1).mean():.1%} lyrics need >1 chunk, {time.time() - t0:.0f}s')
    return out

np.save(os.path.join(ENC_DIR, 'text_sbert_mpnet_trunc.npy'),
        np.vstack(lyrics_df['lyrics_sbert'].values).astype(np.float32))
E_mpnet  = st_chunked('sentence-transformers/all-mpnet-base-v2', 'sbert_mpnet_chunk')
E_minilm = st_chunked('sentence-transformers/all-MiniLM-L6-v2', 'minilm_chunk')
E_bge    = st_chunked('BAAI/bge-base-en-v1.5', 'bge_chunk')

old = l2(np.vstack(lyrics_df['lyrics_sbert'].values).astype(np.float64))
cs = (old * l2(E_mpnet.astype(np.float64))).sum(1)
print(f'cos(truncated mpnet, chunked mpnet): median {np.median(cs):.3f}, 5th pct {np.percentile(cs, 5):.3f}')

sbert_mpnet_chunk: exists
minilm_chunk: exists


bge_chunk: (2075, 768), max_seq=512, chunks/lyric mean 1.17, 14.1% lyrics need >1 chunk, 558s
cos(truncated mpnet, chunked mpnet): median 1.000, 5th pct 0.937


## 2. Emotion-tuned DistilRoBERTa (embedding + 7 emotion probabilities)

In [3]:
EMO = 'j-hartmann/emotion-english-distilroberta-base'
p_emb, p_prob = os.path.join(ENC_DIR, 'text_emoroberta_emb.npy'), os.path.join(ENC_DIR, 'text_emoroberta_probs.npy')
if os.path.exists(p_emb) and os.path.exists(p_prob):
    E_emo, P_emo = np.load(p_emb), np.load(p_prob)
else:
    tok = AutoTokenizer.from_pretrained(EMO)
    clf = AutoModelForSequenceClassification.from_pretrained(EMO).eval()
    labels = [clf.config.id2label[i] for i in range(clf.config.num_labels)]
    t0 = time.time()
    all_chunks = [chunk(t, tok, 126) for t in texts]
    flat = [c for cs in all_chunks for c, _ in cs]
    embs, probs = [], []
    with torch.no_grad():
        for s in range(0, len(flat), 32):
            x = tok(flat[s:s + 32], padding=True, truncation=True, max_length=128, return_tensors='pt')
            o = clf(**x, output_hidden_states=True)
            m = x['attention_mask'].unsqueeze(-1).float()
            embs.append(((o.hidden_states[-1] * m).sum(1) / m.sum(1)).numpy())
            probs.append(torch.softmax(o.logits, -1).numpy())
    embs, probs = np.vstack(embs), np.vstack(probs)
    E_emo, P_emo, k = [], [], 0
    for cs in all_chunks:
        w = np.array([n for _, n in cs], float)[:, None]
        E_emo.append((l2(embs[k:k + len(cs)]) * w).sum(0) / w.sum())
        P_emo.append((probs[k:k + len(cs)] * w).sum(0) / w.sum()); k += len(cs)
    E_emo, P_emo = np.vstack(E_emo).astype(np.float32), np.vstack(P_emo).astype(np.float32)
    np.save(p_emb, E_emo); np.save(p_prob, P_emo)
    json.dump(labels, open(os.path.join(ENC_DIR, 'text_emoroberta_labels.json'), 'w'))
    print(f'emoroberta: {len(flat)} chunks in {time.time() - t0:.0f}s')
labels = json.load(open(os.path.join(ENC_DIR, 'text_emoroberta_labels.json')))
print('labels:', labels)
q = merged.loc[sids, 'quadrant'].values
print('Mean emotion probability by quadrant (evaluation-only sanity check):')
print(pd.DataFrame(P_emo, columns=labels).groupby(q).mean().round(3).to_string())

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

emoroberta: 6207 chunks in 300s
labels: ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
Mean emotion probability by quadrant (evaluation-only sanity check):
    anger  disgust   fear    joy  neutral  sadness  surprise
Q1  0.136    0.057  0.147  0.183    0.233    0.140     0.103
Q2  0.362    0.112  0.227  0.021    0.098    0.142     0.038
Q3  0.111    0.054  0.167  0.044    0.129    0.417     0.078
Q4  0.075    0.037  0.161  0.220    0.188    0.246     0.073


## 3. Lexicon baseline (existing Phase 3 features)

In [4]:
st = lyrics_df['lyrics_structural'].map(lambda s: json.loads(s) if isinstance(s, str) else s)
lex = np.hstack([np.vstack(lyrics_df['lyrics_nrc'].values),
                 np.c_[st.map(lambda d: d['vader_compound']).values, st.map(lambda d: d['lexical_diversity']).values]]).astype(np.float32)
np.save(os.path.join(ENC_DIR, 'text_lexicon.npy'), lex)
print('lexicon', lex.shape, 'NaN:', np.isnan(lex).any())
for f in sorted(os.listdir(ENC_DIR)):
    if f.startswith('text_') and f.endswith('.npy'):
        print(f'  {f:32s} {np.load(os.path.join(ENC_DIR, f)).shape}')

lexicon (2075, 12) NaN: False
  text_bge_chunk.npy               (2075, 768)
  text_emoroberta_emb.npy          (2075, 768)
  text_emoroberta_probs.npy        (2075, 7)
  text_lexicon.npy                 (2075, 12)
  text_minilm_chunk.npy            (2075, 384)
  text_sbert_mpnet_chunk.npy       (2075, 768)
  text_sbert_mpnet_trunc.npy       (2075, 768)
